In [0]:
incoming_orders = [
    (1002, 102, 502, 1, 750.0, "2026-06-24 09:00:00"),
    (1003, 103, 503, 2, 900.0, "2026-06-24 09:05:00")
]

columns = [
    "order_id",
    "customer_id",
    "product_id",
    "quantity",
    "amount",
    "event_time"
]

incoming_df = spark.createDataFrame(incoming_orders, columns)

display(incoming_df)

order_id,customer_id,product_id,quantity,amount,event_time
1002,102,502,1,750.0,2026-06-24 09:00:00
1003,103,503,2,900.0,2026-06-24 09:05:00


In [0]:
from delta.tables import DeltaTable

silver_table = DeltaTable.forName(
    spark,
    "ecommerce_project.silver_orders"
)

In [0]:
from pyspark.sql.functions import current_timestamp

incoming_df = (
    incoming_df
        .withColumn(
            "processed_timestamp",
            current_timestamp()
        )
)

In [0]:
(
    silver_table.alias("target")
    .merge(
        incoming_df.alias("source"),
        "target.order_id = source.order_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("show tables in ecommerce_project").show()

+-----------------+-------------+-----------+
|         database|    tableName|isTemporary|
+-----------------+-------------+-----------+
|ecommerce_project|bronze_orders|      false|
|ecommerce_project|silver_orders|      false|
+-----------------+-------------+-----------+



In [0]:
display(
    spark.table("ecommerce_project.silver_orders")
)

order_id,customer_id,product_id,amount,event_time,processed_timestamp
1001,101,501,600,2026-06-23 10:10:00,2026-07-05T14:12:52.008Z
1002,102,502,750,2026-06-24 09:00:00,2026-07-05T14:19:26.985Z
1003,103,503,900,2026-06-24 09:05:00,2026-07-05T14:19:26.985Z
